`(1) Env 환경변수`

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [3]:
import os
from pprint import pprint
from typing import List, Tuple

`(3) langfuse handler 설정`

In [4]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

https://huggingface.co/datasets/allganize/RAG-Evaluation-Dataset-KO

In [ ]:
import pandas as pd
from datasets import load_dataset

# 데이터셋 로드
ds = load_dataset("allganize/RAG-Evaluation-Dataset-KO")


# 여기서는 'test' 데이터만 추출해서 CSV로 저장해보겠습니다.
df = ds['test'].to_pandas()

# CSV 파일로 저장
df.to_csv("data/rag_test_data.csv", index=False, encoding='utf-8-sig')

print("데이터가 'rag_test_data.csv'로 저장되었습니다.")

데이터가 'rag_test_data.csv'로 저장되었습니다.


In [ ]:
# 1. 필수 라이브러리 임포트
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. 데이터 로드 (저장해두신 CSV 파일)
# 파일 경로가 맞는지 꼭 확인하세요!
df = pd.read_csv("data/rag_test_data.csv")

# 3. 청크 분할기 설정 (500자 단위, 50자 중첩)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50
)

# 4. 데이터셋을 청크로 변환 및 Document 객체 생성
documents_with_metadata = []

print("데이터 처리 시작...")

for idx, row in df.iterrows():

    content_with_qa = f"{row['target_answer']}"
    
    # 텍스트를 청크로 분할
    chunks = text_splitter.split_text(content_with_qa)
    
    for chunk in chunks:
        # Document 객체 생성 시 question과 domain을 모두 메타데이터로 관리
        doc = Document(
            page_content=chunk,
            metadata={
                "domain": row.get('domain', 'N/A'),
                "question": row.get('question', 'N/A'), # 질문도 메타데이터에 저장
                "file_name": row.get('target_file_name', 'N/A'),
                "page": row.get('target_page_no', 0)
            }
        )
        documents_with_metadata.append(doc)
# 5. 결과 확인
print(f"완료! 총 {len(documents_with_metadata)}개의 청크가 생성되었습니다.")

# 첫 번째 청크와 메타데이터 출력
print("\n[첫 번째 청크 확인]")
print(f"내용: {documents_with_metadata[0].page_content}")
print(f"메타데이터: {documents_with_metadata[0].metadata}")


데이터 처리 시작...
완료! 총 305개의 청크가 생성되었습니다.

[첫 번째 청크 확인]
내용: 시중은행, 지방은행, 인터넷은행 모두 은행업을 영위하기 위해서는 '은행법' 제8조에 근거해 금융위원회의 인가를 필요로 합니다. 그러나 시중은행, 지방은행, 인터넷은행의 인가 요건 및 절차에는 일부 차이가 존재합니다.

첫째, 최저자본금의 경우, 시중은행은 1,000억원이 필요하며, 지방은행과 인터넷은행은 250억원이 필요합니다. 

둘째, 비금융주력자 주식 보유한도의 차이가 있습니다. 시중은행의 경우 4%이며, 지방은행은 15%, 인터넷은행은 34%입니다.

셋째, 영업구역 및 영업방식도 차이가 존재합니다. 시중은행과 지방은행은 온라인과 오프라인을 모두 활용하며 전국 또는 일부 지역에서 영업할 수 있지만, 인터넷전문은행은 전국에서 영업하지만 오직 온라인으로만 이루어집니다.
메타데이터: {'domain': 'finance', 'question': '시중은행, 지방은행, 인터넷은행의 인가 요건 및 절차에 차이가 있는데 그 차이점은 무엇인가요?', 'file_name': '[별첨] 지방은행의 시중은행 전환시 인가방식 및 절차.pdf', 'page': '4'}


In [18]:
#각 청크에 대한
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini")

# 2. 맥락 요약 프롬프트 정의
prompt = ChatPromptTemplate.from_template("""
이 텍스트는 전체 문서의 일부입니다. 
이 조각이 담고 있는 핵심 정보(예: 시중/지방/인터넷은행의 자본금 비교, 주식 보유 한도, 인가 근거법 등)를 구체적으로 요약해주세요.
이 요약문은 검색 시스템이 정확한 정보를 찾기 위한 '검색 인덱스' 역할을 합니다.

텍스트 조각: {chunk_content}
요약문:
""")

chain = prompt | llm | StrOutputParser()

# 3. 각 청크에 맥락 보강
enriched_documents = []

for doc in documents_with_metadata:
    # LLM에게 요약 요청
    context_summary = chain.invoke({"chunk_content": doc.page_content})
    
    # [요약문] + [본문] 결합
    enriched_content = f"{context_summary}"
    
    # 새로운 Document 객체 생성
    enriched_doc = Document(
        page_content=enriched_content,
        metadata=doc.metadata
    )
    enriched_documents.append(enriched_doc)

print(f"총 {len(enriched_documents)}개의 청크에 맥락이 보강되었습니다.")
print("\n[보강된 청크 예시]:")
print(enriched_documents[0].page_content)

총 305개의 청크에 맥락이 보강되었습니다.

[보강된 청크 예시]:
- **인가 근거법**: 은행법 제8조에 근거하여 금융위원회의 인가 필요.
- **최저자본금**: 
  - 시중은행: 1,000억원
  - 지방은행: 250억원
  - 인터넷은행: 250억원
- **비금융주력자 주식 보유 한도**:
  - 시중은행: 4%
  - 지방은행: 15%
  - 인터넷은행: 34%
- **영업구역 및 방식**:
  - 시중은행 및 지방은행: 온라인과 오프라인, 전국 또는 일부 지역 영업
  - 인터넷은행: 전국 영업, 오직 온라인으로만 운영


In [ ]:
normal_db.delete_collection()
contextual_db.delete_collection()

In [30]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# 2. Chroma 벡터스토어 빌드 및 저장 
# 일반 문서용 벡터 DB (비교군)
normal_db = Chroma.from_documents(
    documents=documents_with_metadata,  # Replace with your document list
    embedding=embeddings,    
    collection_name="normal", 
    persist_directory="./chroma_db",
  
)
#보강된 문서용 벡터 DB (실험군)
contextual_db = Chroma.from_documents(
    documents=enriched_documents,  # Replace with your document list
    embedding=embeddings,    
    collection_name="contextural", 
    persist_directory="./chroma_db",
  
)


In [ ]:

# 3. 테스트할 질문 3개
test_queries = [
    "시중은행의 최저자본금은 얼마야?",
    "대주주가 충족해야 하는 기타 요건",
    "은행업 인가를 받으려면 어떤 법을 근거로 해야 해?"
]

# 4. 성능 비교 실행
for query in test_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    normal_retriever = normal_db.as_retriever(search_kwargs={"k": 3})
    contextual_retriever = contextual_db.as_retriever(search_kwargs={"k": 3})
    # 일반 검색

    normal_results = normal_retriever.invoke(query)
    print(f"\n[일반 검색 결과]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


    # Contextual 검색
    contextual_results = contextual_retriever.invoke(query, config={"metadata": {"domain": "finance"}} ) 
    print(f"\n[Contextual 검색 결과]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


쿼리: '시중은행의 최저자본금은 얼마야?'

[일반 검색 결과]
  1. 시중은행, 지방은행, 인터넷은행 모두 은행업을 영위하기 위해서는 '은행법' 제8조에 근거해 금융위원회의 인가를 필요로 합니다. 그러나 시중은행, 지방은행, 인터넷은행의 인가 요건 ...
  2. 연체이자는 연체이자의 계산기간 동안 해당 연도마다 1월 1일 현재 전국은행이 적용하는 정기예금금리 중 가장 높은 금리의 2배에 해당하는 금리를 적용하여 산정한다. 이 경우 연체이자...
  3. 산업형기금이 3년 평균 9%로 가장 높은 수익률을 보이고 있습니다....

[Contextual 검색 결과]
  1. - **인가 근거법**: 은행법 제8조에 근거하여 금융위원회의 인가 필요.
- **최저자본금**: 
  - 시중은행: 1,000억원
  - 지방은행: 250억원
  - 인터넷은행:...
  2. 2023년 기준으로 시중은행, 지방은행, 인터넷은행의 자본금 비교 및 주식 보유 한도, 인가 근거법에 대한 정보가 포함되어 있습니다. 추가적인 자세한 내용은 문서 본문 참조 필요....
  3. 주요 정보: 
- 시중은행/지방은행/인터넷은행 자본금 비교: 시중은행 24.6%, 지방은행 4.8%
- 주식 보유 한도 및 인가 근거법: 구체적인 내용은 제공되지 않음.

이 정보...

쿼리: '대주주가 충족해야 하는 기타 요건'

[일반 검색 결과]
  1. 은행업을 신청하려면 대주주 요건을 충족해야 합니다. 대주주 요건으로는 부실금융기관 관련 책임이 없어야 하며, 주주구성계획이 은행법상 소유규제에 적합해야 합니다. 대주주 요건을 증명...
  2. 공통적으로 상정된 안건은 제무제표 승인, 정관 변경, 이사 선임, 이사 보수한도액 승인입니다. 2020년 정기 주주총회에서는 이익잉여금 처분계산서 승인, 감사 보수한도액 승인이 추...
  3. 2020년 3월 24일 주주총회에서 원고가 행사한 의결권은 제7기 제무제표, 이익잉여금 처분계산서 승인, 정관 변경, 이사 선임, 이사 보수한도액 승인, 감사 보수

In [66]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# 1. 키워드 검색기 (BM25) 설정
bm25_retriever_normal = BM25Retriever.from_documents(documents_with_metadata, k=3)

bm25_retriever = BM25Retriever.from_documents(enriched_documents, k=3)


# 2. 벡터 검색기 (의미 기반) 설정
vector_retriever_normal = normal_db.as_retriever(search_kwargs={"k": 3})
vector_retriever = contextual_db.as_retriever(search_kwargs={"k": 3})

# 3. 가중치 조합별 테스트
weights_options = [[0.3, 0.7], [0.5, 0.5], [0.7, 0.3]]

for weights in weights_options:
    print(f"\n[가중치 조합 테스트: BM25 {weights[0]} : Vector {weights[1]}]")
    
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=weights
    )

        
    ensemble_retriever_normal = EnsembleRetriever(
        retrievers=[bm25_retriever_normal, vector_retriever_normal],
        weights=weights
    )
    hybrid_queries = [
        "한국에 대한 성과 및 주식시장 선호도의 예상 순서는 어떻게 됩니까?",           # 의미 기반
        "중국 시장 규모",          # 키워드 기반
        "2019년 YTD 기준으로 브라질의 주식 시장 수익률과 베트남의 주식 시장 수익률 사이에 어떤 차이?",     # 혼합
        ]


    for query in hybrid_queries:
        print(f"\n{'='*80}")
        print(f"쿼리: '{query}'")
        print("="*80)
        results = ensemble_retriever.invoke(query)
        results_normal = ensemble_retriever_normal.invoke(query)
        # 검색 수행
        for i, doc in enumerate(results):
            print(f"Top {i+1}: {doc.page_content[:200]}...")
        for i, doc in enumerate(results_normal):
            print(f"normal_Top {i+1}: {doc.page_content[:200]}...")




        


[가중치 조합 테스트: BM25 0.3 : Vector 0.7]

쿼리: '한국에 대한 성과 및 주식시장 선호도의 예상 순서는 어떻게 됩니까?'
Top 1: 국가별 성과: 미국 > 중국 > 한국; 주식시장 선호도: 중국 > 미국 > 한국....
Top 2: 초단기금융시장 내 자산운용사의 영향력 확대가 예상됨에 따라, 자산운용사가 공개시장운영 대상기관으로 선정될 경우 한국은행과의 직접 거래가 가능해진다. 이로 인해 MMF 등에서의 단기자금 공급 규모가 변동하고, 한국은행이 공개시장운영을 통해 초단기금리의 변동성을 완화할 수 있을 것으로 기대된다....
Top 3: 2023년 4/4분기 한국은행은 유동성 조절을 위해 통화안정증권 발행, RP순매각 및 통화안정계정 예치 규모를 줄였으며, 각각 3.4조 원, 5.5조 원, 5.4조 원 감소함. 2024년 1월 중 통화안정증권 발행 잔액은 5.4조 원 감소하고, RP순매각 및 통화안정계정 예치 규모는 각각 4.8조 원 및 0.5조 원 증가함....
Top 4: 보험업권 상생금융 2024년 지원실적 예상 5,200억원....
Top 5: - 변액연금 잔고: 2002년 0.1조엔 → 2012년 19.0조엔 증가
- 월지급식 펀드 순자산: 2002년 2.7조엔 → 2011년 33.2조엔 증가
- 노후 대비 투자 니즈 증가: 사회 보장제도 지속 가능성, 의료비 부담 상승, 젊은 층 중심
- 고령화로 인한 사회 보장비용 증가 예상
- 고령 세대주 가구의 금융자산 및 실물자산 활용 필요
- 약 90...
normal_Top 1: 미국, 중국, 한국에 대한 국가별 성과는 미국, 중국, 한국 순으로 가장 높았고 국가별 주식시장 선호도는 중국, 미국, 한국 순으로 높을 것으로 예상됩니다....
normal_Top 2: 초단기금융시장에서 자산운용사의 영향력이 확대됨에 따라, 자산운용사가 실제 공개시장운영 대상기관으로 선정되면 이들 기관과 한국은행간에 직접적인 거래가 가능해질 것입니다. 따라서, MMF 등으로부터의 단기자금 공급 규모가 

In [67]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# 1. CrossEncoder 모델 객체 생성 (문자열 모델명을 여기에 전달)
encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

# 2. Reranker 객체 생성 (생성한 encoder 객체를 model 인자로 전달)
reranker = CrossEncoderReranker(model=encoder, top_n=3)

# 3. 압축 검색기 설정 (
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, 
    base_retriever=contextual_db.as_retriever(search_kwargs={"k": 5})  
)

compression_retriever_normal = ContextualCompressionRetriever(
    base_compressor=reranker, 
    base_retriever=normal_db.as_retriever(search_kwargs={"k": 5})  
)
hybrid_queries = [
        "한국에 대한 성과 및 주식시장 선호도의 예상 순서는 어떻게 됩니까?",           # 의미 기반
        "중국 이커머스",          # 키워드 기반
        "2019년 YTD 기준으로 브라질의 주식 시장 수익률과 베트남의 주식 시장 수익률 사이에 어떤 차이?",     # 혼합
        ]
# 4. 검색 실행

for query in hybrid_queries:
        print(f"\n{'='*80}")
        print(f"쿼리: '{query}'")
        print("="*80)
        results = compression_retriever.invoke(query)
        for i, doc in enumerate(results):
            print(f"Top {i+1}: {doc.page_content[:100]}...")

        normal_results = compression_retriever_normal.invoke(query)
        for i, doc in enumerate(normal_results):
            print(f"normal_Top {i+1}: {doc.page_content[:100]}...")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


쿼리: '한국에 대한 성과 및 주식시장 선호도의 예상 순서는 어떻게 됩니까?'
Top 1: 국가별 성과: 미국 > 중국 > 한국; 주식시장 선호도: 중국 > 미국 > 한국....
Top 2: 2024년 한국 GDP 성장률은 2.1%로 전망되며, 이는 지난해 11월의 예측과 일치합니다. 설비투자는 글로벌 제조업 경기 부진으로 둔화되었으나, IT경기 회복과 반도체 기업의 ...
Top 3: 초단기금융시장 내 자산운용사의 영향력 확대가 예상됨에 따라, 자산운용사가 공개시장운영 대상기관으로 선정될 경우 한국은행과의 직접 거래가 가능해진다. 이로 인해 MMF 등에서의 단기...
normal_Top 1: 미국, 중국, 한국에 대한 국가별 성과는 미국, 중국, 한국 순으로 가장 높았고 국가별 주식시장 선호도는 중국, 미국, 한국 순으로 높을 것으로 예상됩니다....
normal_Top 2: 2010년부터 2020년까지 한국의 유통시장은 약 2배, 온라인몰 시장은 약 6배 증가하였으므로, 온라인몰 시장이 더 빠르게 성장하였습니다....
normal_Top 3: 초단기금융시장에서 자산운용사의 영향력이 확대됨에 따라, 자산운용사가 실제 공개시장운영 대상기관으로 선정되면 이들 기관과 한국은행간에 직접적인 거래가 가능해질 것입니다. 따라서, M...

쿼리: '중국 이커머스'
Top 1: 중국: 이커머스 시장 규모 2,010조원; 인도: 이커머스 시장 규모 71조원....
Top 2: 2020년 글로벌 이커머스 시장 규모: 4.2조 USD (약 5천조원), 중국의 온라인 소비 비율: 40.5%....
Top 3: 퀵커머스는 속도와 즉시 배송을 주요 경쟁력으로 하고, 이커머스는 할인과 가격 경쟁력을 강조한다....
normal_Top 1: 이커머스 시장 규모가 가장 큰 국가는 중국으로, 그 규모는 2,010조원입니다. 이에 비해 가장 작은 규모를 보인 국가는 인도로, 그 규모는 71조원입니다....
normal_Top 2: 2020년 글로벌 이커머스 시장의 규모는 4.2조 USD(

In [69]:
# 다양한 Retriever 성능 비교 (Kiwi 기반)
from kiwipiepy import Kiwi

retrievers_to_compare = {
    "일반 Embedding": normal_retriever,
    "일반 BM25": bm25_retriever_normal,
    "일반 Hybrid": ensemble_retriever_normal,
    "Contextual Embedding": contextual_retriever,
    "Contextual BM25": bm25_retriever,
    "Contextual Hybrid": ensemble_retriever,
    "Contextual + Reranker": compression_retriever,
}

# 테스트 쿼리와 기대 키워드 (딕셔너리 형태 구조로 수정)
# expected에는 검색된 문서 청크 내에 포함되어 있어야 하는 핵심 키워드를 지정합니다.
test_queries = [
    {
        "query": "한국에 대한 성과 및 주식시장 선호도의 예상 순서는 어떻게 됩니까?", 
        "expected": "미국"  # 문서 검색 시 포함되어야 할 기대 키워드 예시
    },
    {
        "query": "중국 이커머스", 
        "expected": "중국"
    },
    {
        "query": "2019년 YTD 기준으로 브라질의 주식 시장 수익률과 베트남의 주식 시장 수익률 사이에 어떤 차이?", 
        "expected": "브라질"
    },
]

kiwi = Kiwi()

def evaluate_contextual_retrievers(retrievers: dict, test_queries: list, k: int = 1):
    """Contextual vs Non-Contextual Retriever 성능 비교
    
    k=1로 설정하여 가장 관련성 높은 1개 청크만 검색.
    문서가 작을 때 (6개 청크) k=3이면 50%를 검색하므로 변별력이 낮음.
    """
    results_dict = {}
    
    for name, retriever in retrievers.items():
        correct = 0
        total = len(test_queries)
        
        for test in test_queries:
            query = test["query"]
            expected = test["expected"]
            
            try:
                results = retriever.invoke(query)[:k]
                all_content = " ".join([
                    doc.metadata.get('original_content', '') or doc.page_content 
                    for doc in results
                ])
                
                # 기대 키워드가 검색 결과에 포함되어 있는지 확인
                if expected in all_content:
                    correct += 1
            except Exception as e:
                pass
        
        accuracy = correct / total
        results_dict[name] = {"accuracy": accuracy, "correct": correct, "total": total}
    
    return results_dict

print("="*80)
print("Contextual Retrieval 성능 비교 (k=1, Top-1 정확도)")
print("="*80)

eval_results = evaluate_contextual_retrievers(retrievers_to_compare, test_queries, k=1)

for name, result in eval_results.items():
    status = "✅" if result["accuracy"] >= 0.5 else "❌"
    print(f"{status} {name:25s}: {result['accuracy']:.1%} ({result['correct']}/{result['total']})")

# 일반 vs Contextual 비교
print("\n" + "="*80)
print("일반 vs Contextual 비교 요약")
print("="*80)

normal_avg = sum([eval_results[k]["accuracy"] for k in ["일반 Embedding", "일반 BM25", "일반 Hybrid"]]) / 3
contextual_avg = sum([eval_results[k]["accuracy"] for k in ["Contextual Embedding", "Contextual BM25", "Contextual Hybrid"]]) / 3
best_result = eval_results["Contextual + Reranker"]["accuracy"]

print(f"일반 검색 평균 정확도:      {normal_avg:.1%}")
print(f"Contextual 검색 평균 정확도: {contextual_avg:.1%}")
print(f"Contextual + Reranker:      {best_result:.1%}")

if normal_avg > 0:
    improvement = ((contextual_avg - normal_avg) / normal_avg * 100)
    print(f"\n→ Contextual 방식이 일반 방식 대비 {improvement:.1f}% 향상")
else:
    print(f"\n→ 일반 검색 정확도가 0%이므로 비율 계산 불가")


Contextual Retrieval 성능 비교 (k=1, Top-1 정확도)
✅ 일반 Embedding             : 100.0% (3/3)
✅ 일반 BM25                  : 100.0% (3/3)
✅ 일반 Hybrid                : 100.0% (3/3)
✅ Contextual Embedding     : 100.0% (3/3)
✅ Contextual BM25          : 66.7% (2/3)
✅ Contextual Hybrid        : 100.0% (3/3)
✅ Contextual + Reranker    : 100.0% (3/3)

일반 vs Contextual 비교 요약
일반 검색 평균 정확도:      100.0%
Contextual 검색 평균 정확도: 88.9%
Contextual + Reranker:      100.0%

→ Contextual 방식이 일반 방식 대비 -11.1% 향상
